In [ ]:
!pip install -q -U transformers datasets sentence-transformers faiss-cpu rank-bm25 rouge-score sacrebleu bert-score accelerate bitsandbytes evaluate scipy sentencepiece


In [ ]:
%%writefile /kaggle/working/kaggle_adaptive_rag_beir.py
from __future__ import annotations

import argparse
import gc
import json
import math
import os
import re
import string
import time
from dataclasses import dataclass
from pathlib import Path
from statistics import mean
from typing import Any, Dict, Iterable, List, Sequence, Tuple

# Force PyTorch/bitsandbytes to use only one GPU to prevent multi-GPU hangs on Kaggle
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import faiss
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rank_bm25 import BM25Okapi
from rouge_score import rouge_scorer
from sentence_transformers import CrossEncoder, SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import functools
print = functools.partial(print, flush=True)

# =========================================================================
# CONFIGURATION
# =========================================================================
BEIR_DATASETS = [
    "BeIR/fiqa",        # Finance
    "BeIR/scifact",     # Scientific Fact Checking
    "BeIR/nfcorpus",    # Medical/Nutrition
    "BeIR/trec-covid",  # Bio-Medical (COVID)
    "BeIR/nq",          # Natural Questions (Wikipedia)
    "BeIR/hotpotqa",    # Multi-hop reasoning
    "BeIR/quora",       # Quora Duplicate questions
    "BeIR/fever",       # Fact checking
    "BeIR/scidocs",     # Scientific documents
    "BeIR/arguana"      # Argument retrieval
]
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EVAL_GEN_MODEL = "sentence-transformers/paraphrase-mpnet-base-v2"
OUTPUT_ROOT = Path("evaluation_outputs")

VANILLA_MODELS = ["unsloth/Qwen2.5-7B-Instruct-bnb-4bit"]

# These are set explicitly in main() — NOT as side effects inside constructors
EVAL_EMBEDDER = None
EVAL_EMBEDDER_GENERATIVE = None
ROUGE_SCORER_OBJ = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
BLEU_SMOOTHING = SmoothingFunction().method1


def configure_runtime() -> None:
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True


def normalize_text(text: str) -> str:
    text = str(text or "").lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())


def tokenize_for_bm25(text: str) -> List[str]:
    return re.findall(r"[a-z0-9_+-]+", str(text or "").lower())


# =========================================================================
# EVALUATION METRICS HELPERS
# =========================================================================

# ---- RETRIEVAL METRICS ----

def compute_ndcg_at_k(retrieved_ids: List[str], relevant_ids: set, k: int) -> float:
    dcg = 0.0
    for idx, doc_id in enumerate(retrieved_ids[:k]):
        if doc_id in relevant_ids:
            dcg += 1.0 / math.log2(idx + 2)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(k, len(relevant_ids))))
    return dcg / idcg if idcg > 0.0 else 0.0


def compute_mrr_at_k(retrieved_ids: List[str], relevant_ids: set, k: int) -> float:
    for rank, doc_id in enumerate(retrieved_ids[:k], 1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def compute_recall_at_k(retrieved_ids: List[str], relevant_ids: set, k: int) -> float:
    retrieved_set = set(retrieved_ids[:k])
    intersection = retrieved_set & relevant_ids
    return len(intersection) / len(relevant_ids) if len(relevant_ids) > 0 else 0.0


def compute_precision_at_k(retrieved_ids: List[str], relevant_ids: set, k: int) -> float:
    """Precision@K = |retrieved ∩ relevant| / K"""
    retrieved_set = set(retrieved_ids[:k])
    intersection = retrieved_set & relevant_ids
    return len(intersection) / k if k > 0 else 0.0


def compute_hit_rate_at_k(retrieved_ids: List[str], relevant_ids: set, k: int) -> float:
    """Hit Rate@K = 1 if any relevant doc in top-K, else 0"""
    return 1.0 if set(retrieved_ids[:k]) & relevant_ids else 0.0


# ---- GENERATION METRICS ----

def compute_token_f1(pred: str, gold: str) -> float:
    pred_tokens = tokenize_for_bm25(pred)
    gold_tokens = tokenize_for_bm25(gold)
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = set(pred_tokens) & set(gold_tokens)
    num_same = sum(min(pred_tokens.count(w), gold_tokens.count(w)) for w in common)
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * (precision * recall) / (precision + recall)


def compute_bleu(pred: str, gold: str) -> float:
    """BLEU score with smoothing (handles short predictions)."""
    pred_tokens = tokenize_for_bm25(pred)
    gold_tokens = tokenize_for_bm25(gold)
    if not pred_tokens or not gold_tokens:
        return 0.0
    return sentence_bleu([gold_tokens], pred_tokens, smoothing_function=BLEU_SMOOTHING)


def compute_exact_match(pred: str, gold: str) -> float:
    """Exact match after normalisation (lowercasing, punctuation removal, whitespace collapse)."""
    return 1.0 if normalize_text(pred) == normalize_text(gold) else 0.0


# =========================================================================
# ECE — Expected Calibration Error (Guo et al. 2017)
# =========================================================================
def compute_ece(confidence_scores, actual_hits, n_bins=5):
    """
    Expected Calibration Error.
    Measures gap between predicted confidence and actual accuracy.
    Perfect calibration = ECE of 0.0
    """
    confidence_scores = np.array(confidence_scores)
    actual_hits = np.array(actual_hits)

    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    diagram_data = []

    for i in range(n_bins):
        mask = (confidence_scores >= bins[i]) & (confidence_scores < bins[i + 1])
        if mask.sum() == 0:
            continue
        avg_conf = float(confidence_scores[mask].mean())
        avg_acc = float(actual_hits[mask].mean())
        weight = mask.sum() / len(confidence_scores)
        ece += weight * abs(avg_conf - avg_acc)
        diagram_data.append({
            "bin_lower": round(float(bins[i]), 2),
            "bin_upper": round(float(bins[i + 1]), 2),
            "avg_confidence": round(avg_conf, 4),
            "avg_accuracy": round(avg_acc, 4),
            "count": int(mask.sum())
        })

    return ece, diagram_data


# =========================================================================
# Bootstrap Confidence Intervals
# =========================================================================
def bootstrap_ci(values, n_bootstrap=1000, ci=95):
    """95% confidence interval via bootstrap resampling."""
    values = np.array(values)
    if len(values) == 0:
        return 0.0, 0.0, 0.0
    rng = np.random.RandomState(42)
    means = [
        np.mean(rng.choice(values, len(values), replace=True))
        for _ in range(n_bootstrap)
    ]
    lower = np.percentile(means, (100 - ci) / 2)
    upper = np.percentile(means, 100 - (100 - ci) / 2)
    return round(float(np.mean(values)), 4), round(float(lower), 4), round(float(upper), 4)


# =========================================================================
# HETEROGENEOUS DYNAMIC DATASET LOADER (Ensures Gold Documents are present)
# =========================================================================
def load_and_build_beir_data(dataset_names: List[str], limit: int, corpus_size: int):
    all_docs = []
    all_samples = []

    # Dynamically allocate requested limits across all datasets evenly
    limit_per_ds = max(1, limit // len(dataset_names)) if limit > 0 else 0
    corpus_size_per_ds = max(10, corpus_size // len(dataset_names))

    successful_datasets = []

    for ds_name in dataset_names:
        print(f"\n[Dataset] Attempting to load domain: {ds_name}...")
        
        try:
            corpus_ds = load_dataset(ds_name, "corpus", split="corpus")
            queries_ds = load_dataset(ds_name, "queries", split="queries")
            
            qrels_name = f"{ds_name}-qrels"
            try:
                qrels_ds = load_dataset(qrels_name, split="test")
            except Exception:
                try:
                    qrels_ds = load_dataset(qrels_name, split="validation")
                except Exception:
                    qrels_ds = load_dataset(qrels_name, split="train")
        except Exception as e:
            print(f"  -> [Warning] Failed to load {ds_name}. Skipping. Error: {e}")
            continue

        prefix = ds_name.split("/")[-1] + "_" # Add prefix to prevent ID collisions

        corpus_dict = {}
        corpus_docs_list = []
        for row in corpus_ds:
            doc_id_key = "_id" if "_id" in row else "id"
            doc_id = prefix + str(row[doc_id_key])
            doc_item = {
                "doc_id": doc_id,
                "title": str(row.get("title", doc_id)),
                "text": str(row.get("text", "")).strip(),
                "source_dataset": ds_name
            }
            corpus_dict[doc_id] = doc_item
            corpus_docs_list.append(doc_item)

        queries_dict = {}
        for row in queries_ds:
            q_id_key = "_id" if "_id" in row else "id"
            queries_dict[prefix + str(row[q_id_key])] = str(row.get("text", ""))

        qrels_dict = {}
        for row in qrels_ds:
            qid_key = "query-id" if "query-id" in row else "query_id"
            cid_key = "corpus-id" if "corpus-id" in row else "corpus_id"
            score_key = "score" if "score" in row else ("relevance" if "relevance" in row else None)
            
            qid = prefix + str(row[qid_key])
            cid = prefix + str(row[cid_key])
            score = int(row[score_key]) if score_key and row[score_key] is not None else 1
            
            if score > 0:
                if qid not in qrels_dict:
                    qrels_dict[qid] = set()
                qrels_dict[qid].add(cid)

        ds_samples = []
        for qid, relevant_ids in qrels_dict.items():
            if qid in queries_dict:
                valid_rel_ids = {cid for cid in relevant_ids if cid in corpus_dict}
                if valid_rel_ids:
                    first_rel_id = list(valid_rel_ids)[0]
                    ds_samples.append({
                        "qid": qid,
                        "source_dataset": ds_name,
                        "question": queries_dict[qid],
                        "answer": corpus_dict[first_rel_id]["text"],
                        "relevant_ids": valid_rel_ids,
                    })

        selected_samples = ds_samples[:limit_per_ds] if limit_per_ds > 0 else ds_samples

        relevant_doc_ids = set()
        for s in selected_samples:
            relevant_doc_ids.update(s["relevant_ids"])

        docs = []
        added_doc_ids = set()

        for doc_id in relevant_doc_ids:
            if doc_id in corpus_dict:
                docs.append(corpus_dict[doc_id])
                added_doc_ids.add(doc_id)

        for doc_item in corpus_docs_list:
            if len(docs) >= corpus_size_per_ds:
                break
            if doc_item["doc_id"] not in added_doc_ids:
                docs.append(doc_item)
                added_doc_ids.add(doc_item["doc_id"])

        all_docs.extend(docs)
        all_samples.extend(selected_samples)
        successful_datasets.append(ds_name)
        print(f"  -> Success: Added {len(selected_samples)} queries and {len(docs)} corpus docs.")

        # Aggressive RAM Management to prevent Kaggle OOM crash
        del corpus_ds, queries_ds, qrels_ds, corpus_dict, corpus_docs_list
        gc.collect()

    print(f"\n[Dataset] HETEROGENEOUS MERGE COMPLETE.")
    print(f"Successfully loaded domains: {', '.join(successful_datasets)}")
    print(f"Total queries: {len(all_samples)} | Total corpus: {len(all_docs)}")
    return all_docs, all_samples


# =========================================================================
# RETRIEVERS
# =========================================================================
def encode_texts(model, texts, batch_size=64):
    if not texts:
        dim = model.get_sentence_embedding_dimension()
        return np.zeros((0, dim), dtype="float32")
    return np.asarray(
        model.encode(
            list(texts),
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        ),
        dtype="float32"
    )


class DenseRetriever:
    def __init__(self, docs: Sequence[Dict[str, Any]], batch_size: int = 96) -> None:
        self.docs = list(docs)
        self.embedder = SentenceTransformer(EMBED_MODEL)
        # No global side effects here — globals set in main()
        print("[DenseRetriever] Encoding corpus embeddings...")
        embeddings = encode_texts(
            self.embedder,
            [d["text"] for d in self.docs],
            batch_size=batch_size
        )
        print("[DenseRetriever] Embeddings complete.")
        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(embeddings)

    def retrieve(self, query: str, top_k: int) -> List[Dict[str, Any]]:
        query_vec = encode_texts(self.embedder, [query], batch_size=1)
        scores, idxs = self.index.search(query_vec, top_k)
        return [
            {"retrieval_score": float(s), **self.docs[i]}
            for s, i in zip(scores[0], idxs[0])
            if i >= 0
        ]


class BM25Retriever:
    def __init__(self, docs: Sequence[Dict[str, Any]]) -> None:
        self.docs = list(docs)
        self.bm25 = BM25Okapi([tokenize_for_bm25(d["text"]) for d in self.docs])

    def retrieve(self, query: str, top_k: int) -> List[Dict[str, Any]]:
        scores = self.bm25.get_scores(tokenize_for_bm25(query))
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [{"retrieval_score": float(scores[i]), **self.docs[i]} for i in top_indices]


class HybridRetriever:
    def __init__(self, docs: Sequence[Dict[str, Any]], dense_retriever: DenseRetriever = None, batch_size: int = 64) -> None:
        self.docs = list(docs)
        self.dense = dense_retriever if dense_retriever is not None else DenseRetriever(docs, batch_size=batch_size)
        self.bm25 = BM25Retriever(docs)
        self.reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)

    def retrieve(self, query: str, top_k_per_stage: int, rerank_top_k: int) -> List[Dict[str, Any]]:
        dense_docs = self.dense.retrieve(query, top_k_per_stage)
        bm25_docs = self.bm25.retrieve(query, top_k_per_stage)

        all_ids = set([d["doc_id"] for d in dense_docs] + [d["doc_id"] for d in bm25_docs])
        d_ranks = {d["doc_id"]: i for i, d in enumerate(dense_docs)}
        b_ranks = {d["doc_id"]: i for i, d in enumerate(bm25_docs)}

        fused = {doc_id: (1 / (60 + d_ranks.get(doc_id, 1000))) + (1 / (60 + b_ranks.get(doc_id, 1000))) for doc_id in all_ids}
        top_ids = sorted(fused, key=fused.get, reverse=True)[:top_k_per_stage]

        id_to_doc = {d["doc_id"]: d for d in self.docs}
        candidates = [id_to_doc[doc_id] for doc_id in top_ids]

        scores = self.reranker.predict([[query, d["text"]] for d in candidates])
        ranked = [d for d, s in sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)]
        return ranked[:rerank_top_k]


# =========================================================================
# RELIABILITY PREDICTOR (Novelty Module)
# =========================================================================
STOPWORDS = {"a", "an", "the", "and", "or", "but", "is", "are", "what", "how", "why", "in", "on", "for"}


def get_raw_signals(query: str, docs: List[Dict[str, Any]], embedder: SentenceTransformer) -> tuple:
    if not docs:
        return 0.0, 0.0, 0.0

    magnitude = float(np.mean([float(d.get("retrieval_score", 0.0)) for d in docs]))

    doc_embs = encode_texts(embedder, [d["text"] for d in docs], batch_size=len(docs))
    cohesion = magnitude
    if len(docs) > 1:
        sim_matrix = np.dot(doc_embs, doc_embs.T)
        cohesion = float(np.mean(sim_matrix[np.triu_indices(len(docs), k=1)]))

    query_kw = set(t for t in tokenize_for_bm25(query) if t not in STOPWORDS)
    kw_score = sum(1 for d in docs if query_kw & set(tokenize_for_bm25(d["text"]))) / len(docs) if docs and query_kw else 0.0

    return magnitude, cohesion, kw_score


def compute_reliability_score(magnitude, cohesion, kw_score, weights: dict) -> float:
    """Old weighted-sum reliability score. Used by CRAG variant (fixed weights)."""
    rel = (weights["mag"] * magnitude) + (weights["coh"] * cohesion) + (weights["kw"] * kw_score)
    return max(0.0, min(1.0, rel))


def get_calibrated_prob(calibrator, mag, coh, kw):
    """Platt-scaled probability from raw signals."""
    clf, scaler = calibrator
    X = scaler.transform([[mag, coh, kw]])
    return float(clf.predict_proba(X)[0][1])


# =========================================================================
# PLATT SCALING CALIBRATION
# =========================================================================
def calibrate_system(val_samples: List[Dict], dense: DenseRetriever, hybrid: HybridRetriever, output_dir: Path):
    print("\n[Calibration] Running Platt Scaling Calibration...")
    X, y = [], []

    for s in tqdm(val_samples, desc="Calibration signals"):
        docs = dense.retrieve(s["question"], top_k=3)
        mag, coh, kw = get_raw_signals(s["question"], docs, dense.embedder)
        hit = 1 if any(d["doc_id"] in s["relevant_ids"] for d in docs) else 0
        X.append([mag, coh, kw])
        y.append(hit)

    X = np.array(X)
    y = np.array(y)

    # Safety check: LogisticRegression needs both classes (0 and 1).
    if len(set(y)) < 2:
        print(f"[Calibration] WARNING: Only one class in calibration labels (all {'hits' if y[0]==1 else 'misses'}).")
        print(f"  Platt Scaling requires both classes. Falling back to default config.")
        print(f"  This is expected with tiny test runs. Use --limit 300 --corpus_size 3000 for real evaluation.")
        thresholds = {"fast": 0.65, "abstain": 0.35}

        class _DummyCalibrator:
            def __init__(self, fixed_prob):
                self._p = fixed_prob
            def transform(self, X):
                return X
            def predict_proba(self, X):
                n = len(X)
                return np.column_stack([np.full(n, 1-self._p), np.full(n, self._p)])

        fixed_prob = 0.8 if y[0] == 1 else 0.2
        dummy_clf = _DummyCalibrator(fixed_prob)
        dummy_scaler = _DummyCalibrator(fixed_prob)

        calib_info = {
            "method": "Fallback (single-class calibration data)",
            "n_calibration_samples": len(val_samples),
            "unique_labels": int(len(set(y))),
            "fixed_probability": fixed_prob,
            "threshold_fast": thresholds["fast"],
            "threshold_abstain": thresholds["abstain"]
        }
        with open(output_dir / "calibration_info.json", "w") as f:
            json.dump(calib_info, f, indent=2)

        return (dummy_clf, dummy_scaler), thresholds

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    clf = LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    )
    clf.fit(X_scaled, y)

    print(f"[Calibration] Platt Scaling learned coefficients:")
    print(f"  Magnitude : {clf.coef_[0][0]:.4f}")
    print(f"  Cohesion  : {clf.coef_[0][1]:.4f}")
    print(f"  KW Score  : {clf.coef_[0][2]:.4f}")
    print(f"  Intercept : {clf.intercept_[0]:.4f}")

    X_probs = clf.predict_proba(X_scaled)[:, 1]
    ece_val, _ = compute_ece(X_probs, y)
    print(f"[Calibration] ECE on calibration set: {ece_val:.4f}")
    print(f"  (Note: test ECE is what matters — this is optimistic)")

    thresholds = {"fast": 0.65, "abstain": 0.35}
    print(f"[Calibration] Probability thresholds: Fast>={thresholds['fast']}, Abstain<{thresholds['abstain']}")

    calib_info = {
        "method": "Platt Scaling (Logistic Regression)",
        "coef_magnitude": round(float(clf.coef_[0][0]), 4),
        "coef_cohesion": round(float(clf.coef_[0][1]), 4),
        "coef_kw_score": round(float(clf.coef_[0][2]), 4),
        "intercept": round(float(clf.intercept_[0]), 4),
        "n_calibration_samples": len(val_samples),
        "ece_on_calib_set": round(float(ece_val), 4),
        "threshold_fast": thresholds["fast"],
        "threshold_abstain": thresholds["abstain"]
    }
    with open(output_dir / "calibration_info.json", "w") as f:
        json.dump(calib_info, f, indent=2)
    print(f"[Calibration] Coefficients saved to calibration_info.json")

    return (clf, scaler), thresholds


# =========================================================================
# ABLATION STUDY — Proves each signal contributes independently
# =========================================================================
def run_ablation_study(val_samples: List[Dict], dense: DenseRetriever, output_dir: Path):
    print("\n[Ablation] Running Signal Ablation Study...")

    X_full, y = [], []
    for s in val_samples:
        docs = dense.retrieve(s["question"], top_k=3)
        mag, coh, kw = get_raw_signals(s["question"], docs, dense.embedder)
        hit = 1 if any(d["doc_id"] in s["relevant_ids"] for d in docs) else 0
        X_full.append([mag, coh, kw])
        y.append(hit)

    X_full = np.array(X_full)
    y = np.array(y)

    if len(set(y)) < 2:
        print("[Ablation] WARNING: Only one class in data. Skipping ablation.")
        df = pd.DataFrame([{"Signal Subset": "SKIPPED", "Accuracy": "N/A", "ECE": "N/A"}])
        df.to_csv(output_dir / "ablation_study.csv", index=False)
        return df

    subsets = {
        "Magnitude Only":   [0],
        "Cohesion Only":    [1],
        "KW Score Only":    [2],
        "Mag + Cohesion":   [0, 1],
        "Mag + KW":         [0, 2],
        "Coh + KW":         [1, 2],
        "All Three (Full)": [0, 1, 2],
    }

    ablation_rows = []
    for name, indices in subsets.items():
        X_sub = X_full[:, indices]
        scaler_sub = StandardScaler()
        X_scaled = scaler_sub.fit_transform(X_sub)

        clf_sub = LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=42
        )
        clf_sub.fit(X_scaled, y)

        probs = clf_sub.predict_proba(X_scaled)[:, 1]
        ece_val, _ = compute_ece(probs, y)
        acc = clf_sub.score(X_scaled, y)

        ablation_rows.append({
            "Signal Subset": name,
            "Accuracy": round(acc, 4),
            "ECE": round(float(ece_val), 4),
        })
        print(f"  {name}: Acc={acc:.4f}, ECE={ece_val:.4f}")

    df = pd.DataFrame(ablation_rows)
    df.to_csv(output_dir / "ablation_study.csv", index=False)
    print(f"[Ablation] Saved to ablation_study.csv")
    return df


# =========================================================================
# RISK-COVERAGE ANALYSIS — Sweep thresholds to show accuracy-coverage trade-off
# =========================================================================
def compute_risk_coverage(results: List[Dict], output_dir: Path):
    """
    Sweeps confidence thresholds from 0.0 to 1.0 and computes:
      - Coverage: fraction of queries with reliability >= threshold
      - Risk: 1 - avg_F1 on covered queries (lower is better)
    Outputs a CSV for plotting the Risk-Coverage curve.
    """
    print("\n[Risk-Coverage] Computing risk-coverage curve...")
    scored = [r for r in results if r["reliability_score"] > 0]
    if not scored:
        print("[Risk-Coverage] No scored results. Skipping.")
        return

    thresholds_list = np.arange(0.0, 1.01, 0.05)
    rows = []
    for t in thresholds_list:
        covered = [r for r in scored if r["reliability_score"] >= t]
        coverage = len(covered) / len(scored)
        if covered:
            non_abstained = [r for r in covered if not r.get("abstained", False)]
            if non_abstained:
                avg_f1 = float(np.mean([r["f1_score"] for r in non_abstained]))
                avg_ndcg = float(np.mean([r["ndcg_at_3"] for r in non_abstained]))
            else:
                avg_f1 = 0.0
                avg_ndcg = 0.0
            risk = 1.0 - avg_f1
        else:
            risk = 1.0
            avg_f1 = 0.0
            avg_ndcg = 0.0
        rows.append({
            "threshold": round(float(t), 2),
            "coverage": round(coverage, 4),
            "risk": round(risk, 4),
            "avg_f1": round(avg_f1, 4),
            "avg_ndcg": round(avg_ndcg, 4),
            "n_covered": len(covered),
        })

    df = pd.DataFrame(rows)
    df.to_csv(output_dir / "risk_coverage.csv", index=False)
    print(f"[Risk-Coverage] Saved to risk_coverage.csv ({len(rows)} threshold points)")
    return df


# =========================================================================
# LLM GENERATION
# =========================================================================
def build_rag_prompt(question: str, docs: Sequence[Dict[str, Any]], max_docs: int = 3) -> str:
    ctx = "\n\n".join([f"[Doc]\n{d['text'][:800]}" for d in list(docs)[:max_docs]])
    return f"Context:\n{ctx}\n\nQuestion: {question}\nAnswer:"


def load_causal_lm(model_name: str, use_4bit: bool):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    kwargs = {"device_map": "auto", "torch_dtype": torch.float16}

    if use_4bit or "bnb-4bit" in model_name.lower():
        print(f"[LLM] Loading {model_name} with explicit 4-bit BitsAndBytesConfig...")
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )

    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    return tokenizer, model.eval()


def generate_answer(tokenizer, model, prompt: str) -> str:
    messages = [
        {"role": "system", "content": "You are a factual assistant. Answer strictly using the context provided."},
        {"role": "user", "content": prompt}
    ]
    try:
        formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        formatted_prompt = prompt

    device = "cuda" if torch.cuda.is_available() else "cpu"
    encoded = tokenizer(formatted_prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)

    with torch.no_grad():
        out = model.generate(
            **encoded,
            max_new_tokens=100,
            max_length=None,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True
        )

    new_tokens = out[0][encoded["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# =========================================================================
# EVALUATION RUNNER
# =========================================================================
def run_evaluation(
    model_name: str,
    test_samples: List[Dict],
    dense: DenseRetriever,
    bm25: BM25Retriever,
    hybrid: HybridRetriever,
    tokenizer,
    model,
    calibrator,
    thresholds: dict
) -> List[Dict]:

    print(f"\n========== RUNNING RAG VARIANT: {model_name.upper()} ==========")
    results = []

    for idx, s in enumerate(tqdm(test_samples, desc=f"Evaluating {model_name}")):
        q = s["question"]
        gold = s["answer"]
        rel_ids = s["relevant_ids"]
        source_dataset = s.get("source_dataset", "unknown")
        
        start_time = time.time()    

        abstained = False
        pathway = "Direct"
        reliability = 0.0
        docs_for_shadow = []  # Initialised BEFORE routing logic

        # =============================================================
        # 1. RETRIEVAL & PATHWAY ROUTING
        # =============================================================
        if model_name == "vanilla":
            docs = dense.retrieve(q, top_k=3)

        elif model_name == "bm25":
            docs = bm25.retrieve(q, top_k=3)

        elif model_name == "hybrid":
            docs = hybrid.retrieve(q, top_k_per_stage=10, rerank_top_k=3)

        elif model_name == "crag":
            docs = dense.retrieve(q, top_k=3)
            mag, coh, kw = get_raw_signals(q, docs, dense.embedder)
            reliability = compute_reliability_score(
                mag, coh, kw,
                {"mag": 0.5, "coh": 0.3, "kw": 0.2}
            )
            if reliability < 0.5:
                docs = hybrid.retrieve(q, top_k_per_stage=10, rerank_top_k=3)
                pathway = "Corrective"
            else:
                pathway = "Fast"

        elif model_name == "adaptive":
            docs = dense.retrieve(q, top_k=3)
            docs_for_shadow = docs  # Save BEFORE clearing for shadow generation
            mag, coh, kw = get_raw_signals(q, docs, dense.embedder)
            reliability = get_calibrated_prob(calibrator, mag, coh, kw)
            if reliability >= thresholds["fast"]:
                pathway = "Fast"
            elif thresholds["abstain"] <= reliability < thresholds["fast"]:
                docs = hybrid.retrieve(q, top_k_per_stage=10, rerank_top_k=3)
                pathway = "Corrective"
            else:
                pathway = "Abstain"
                abstained = True
                docs = []

        else:
            raise ValueError(f"Unknown variant: {model_name}")

        # =============================================================
        # 2. GENERATION + SHADOW GENERATION FOR ABSTAINED QUERIES
        # =============================================================
        abstain_was_correct = None
        hypothetical_f1 = None

        if abstained:
            pred = "System Abstention: No reliable context found."
            if docs_for_shadow:
                shadow_prompt = build_rag_prompt(q, docs_for_shadow)
                shadow_pred = generate_answer(tokenizer, model, shadow_prompt)
                hypothetical_f1 = compute_token_f1(shadow_pred, gold)
                abstain_was_correct = 1 if hypothetical_f1 < 0.25 else 0
        else:
            prompt = build_rag_prompt(q, docs)
            pred = generate_answer(tokenizer, model, prompt)

        latency = time.time() - start_time

        # =============================================================
        # 3. METRIC COMPUTATION
        # =============================================================

        # --- RETRIEVAL METRICS ---
        # For abstained queries, use docs retrieved BEFORE abstention.
        # Retrieval quality should be measured fairly — the retrieval
        # happened, only generation was suppressed.
        if abstained and docs_for_shadow:
            retrieved_ids = [str(d["doc_id"]) for d in docs_for_shadow]
        else:
            retrieved_ids = [str(d["doc_id"]) for d in docs] if docs else []

        ndcg = compute_ndcg_at_k(retrieved_ids, rel_ids, k=3)
        mrr = compute_mrr_at_k(retrieved_ids, rel_ids, k=3)
        recall = compute_recall_at_k(retrieved_ids, rel_ids, k=3)
        precision = compute_precision_at_k(retrieved_ids, rel_ids, k=3)
        hit_rate = compute_hit_rate_at_k(retrieved_ids, rel_ids, k=3)

        # --- GENERATION METRICS (only on answered queries) ---
        if abstained:
            semsim = 0.0
            rouge_l = 0.0
            f1 = 0.0
            bleu = 0.0
            exact_match = 0.0
            faithfulness = 0.0
            hallucination_rate = 1.0
        else:
            # Semantic Similarity (Answer Relevancy) — independent evaluator
            semsim = float(np.dot(
                EVAL_EMBEDDER_GENERATIVE.encode(pred, normalize_embeddings=True),
                EVAL_EMBEDDER_GENERATIVE.encode(gold, normalize_embeddings=True)
            )) if pred and gold else 0.0

            rouge_l = ROUGE_SCORER_OBJ.score(gold, pred)['rougeL'].fmeasure
            f1 = compute_token_f1(pred, gold)
            bleu = compute_bleu(pred, gold)
            exact_match = compute_exact_match(pred, gold)

            # Faithfulness: cosine sim between answer and RETRIEVED context
            # Measures whether the answer is grounded in the provided documents
            context_text = " ".join([d["text"][:800] for d in docs[:3]])
            faithfulness = float(np.dot(
                EVAL_EMBEDDER_GENERATIVE.encode(pred, normalize_embeddings=True),
                EVAL_EMBEDDER_GENERATIVE.encode(context_text, normalize_embeddings=True)
            )) if pred and context_text else 0.0

            # Hallucination Rate = 1 - Faithfulness (higher = more hallucination)
            hallucination_rate = max(0.0, 1.0 - faithfulness)

        results.append({
            "rag_variant": model_name,
            "qid": s["qid"],
            "source_dataset": source_dataset,
            "question": q,
            "prediction": pred,
            "gold_answer": gold,
            "pathway": pathway,
            "reliability_score": reliability,
            # Retrieval
            "ndcg_at_3": ndcg,
            "mrr_at_3": mrr,
            "recall_at_3": recall,
            "precision_at_3": precision,
            "hit_rate_at_3": hit_rate,
            # Generation
            "semantic_similarity": semsim,
            "rouge_l": rouge_l,
            "f1_score": f1,
            "bleu": bleu,
            "exact_match": exact_match,
            # Hallucination
            "faithfulness": faithfulness,
            "hallucination_rate": hallucination_rate,
            # Meta
            "latency_seconds": latency,
            "abstained": abstained,
            "abstain_was_correct": abstain_was_correct,
            "hypothetical_f1": hypothetical_f1,
        })        
        if (idx + 1) % 10 == 0:
            print(f"[{model_name}] Completed {idx+1}/{len(test_samples)} queries | Last latency: {latency:.1f}s")
    return results


# =========================================================================
# MAIN
# =========================================================================
def main():
    configure_runtime()
    parser = argparse.ArgumentParser()
    parser.add_argument("--models", nargs="+", default=["vanilla", "bm25", "hybrid", "crag", "adaptive"])
    parser.add_argument("--limit", type=int, default=300, help="Total queries to load")
    parser.add_argument("--corpus_size", type=int, default=3000, help="Truncated corpus size")
    parser.add_argument("--use_4bit", type=lambda x: x.lower() != 'false', default=True)
    parser.add_argument("--output_dir", type=Path, default=OUTPUT_ROOT)
    args = parser.parse_args()

    args.output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Load Data
    corpus, samples = load_and_build_beir_data(BEIR_DATASETS, args.limit, args.corpus_size)

    # 2. Split: 150 calibration + rest for test
    num_calib = 300
    if len(samples) < num_calib + 30:
        num_calib = max(1, len(samples) // 3)
    calibration_samples = samples[:num_calib]
    test_samples = samples[num_calib:]

    print(f"\n[Split] Calibration Set Size: {len(calibration_samples)} queries")
    print(f"[Split] Test Set Size: {len(test_samples)} queries")

    # 3. Load LLM & Retrievers
    print("\n[Main] Loading Causal LM...")
    tokenizer, model = load_causal_lm(VANILLA_MODELS[0], use_4bit=args.use_4bit)

    dense_retriever = DenseRetriever(corpus, batch_size=64)
    bm25_retriever = BM25Retriever(corpus)
    hybrid_retriever = HybridRetriever(corpus, dense_retriever=dense_retriever, batch_size=64)

    # Set globals explicitly in main — no side effects in constructors
    global EVAL_EMBEDDER, EVAL_EMBEDDER_GENERATIVE
    EVAL_EMBEDDER = dense_retriever.embedder
    print("[Main] Loading independent generative evaluator (paraphrase-mpnet-base-v2)...")
    EVAL_EMBEDDER_GENERATIVE = SentenceTransformer(EVAL_GEN_MODEL)
    print("[Main] Independent evaluator loaded.")

    # 4. Calibration Phase
    calibrator = None
    thresholds = {"fast": 0.65, "abstain": 0.35}
    if "adaptive" in args.models or "crag" in args.models:
        if len(calibration_samples) > 0:
            calibrator, thresholds = calibrate_system(
                calibration_samples, dense_retriever, hybrid_retriever, args.output_dir
            )
        else:
            print("[Warning] Too few samples for calibration. Using defaults.")

    # 5. Ablation Study — proves each signal contributes independently
    if calibrator is not None:
        ablation_df = run_ablation_study(calibration_samples, dense_retriever, args.output_dir)

    # 6. Run Evaluations on Test Set
    all_runs_results = []
    comparison_rows = []

    for model_name in args.models:
        results = run_evaluation(
            model_name,
            test_samples,
            dense_retriever,
            bm25_retriever,
            hybrid_retriever,
            tokenizer,
            model,
            calibrator,
            thresholds
        )
        all_runs_results.extend(results)

        pd.DataFrame(results).to_csv(args.output_dir / f"{model_name}_detailed.csv", index=False)

        # ----- Metric Aggregation (answered vs abstained split) -----
        answered = [r for r in results if not r["abstained"]]
        abstained_results = [r for r in results if r["abstained"]]

        # Retrieval metrics on ALL queries
        ndcgs = [r["ndcg_at_3"] for r in results]
        mrrs = [r["mrr_at_3"] for r in results]
        recalls = [r["recall_at_3"] for r in results]
        precisions = [r["precision_at_3"] for r in results]
        hit_rates = [r["hit_rate_at_3"] for r in results]

        # Generative metrics ONLY on answered queries
        semsims = [r["semantic_similarity"] for r in answered] if answered else [0.0]
        rouges = [r["rouge_l"] for r in answered] if answered else [0.0]
        f1s = [r["f1_score"] for r in answered] if answered else [0.0]
        bleus = [r["bleu"] for r in answered] if answered else [0.0]
        exact_matches = [r["exact_match"] for r in answered] if answered else [0.0]

        # Hallucination metrics ONLY on answered queries
        faithfulnesses = [r["faithfulness"] for r in answered] if answered else [0.0]
        halluc_rates = [r["hallucination_rate"] for r in answered] if answered else [1.0]

        latencies = [r["latency_seconds"] for r in results]
        abstain_rate = len(abstained_results) / len(results)

        # Abstain Precision and Recall
        abstain_precision = 0.0
        abstain_recall = 0.0
        if abstained_results:
            correct_abstains = [r for r in abstained_results if r["abstain_was_correct"] == 1]
            abstain_precision = len(correct_abstains) / len(abstained_results)

            would_have_failed = [
                r for r in results
                if (r["abstained"] and r["abstain_was_correct"] == 1)
                or (not r["abstained"] and r["f1_score"] < 0.25)
            ]
            abstain_recall = len(correct_abstains) / len(would_have_failed) if would_have_failed else 0.0

        # ECE, Brier Score, AUROC (confidence metrics — only for adaptive/crag)
        ece_score = 0.0
        ece_diagram = []
        brier_score = 0.0
        auroc_score = 0.0
        if model_name in ["adaptive", "crag"]:
            scored_results = [r for r in results if r["reliability_score"] > 0]
            if scored_results:
                conf_scores = np.array([r["reliability_score"] for r in scored_results])
                hit_scores = np.array([1 if r["ndcg_at_3"] > 0 else 0 for r in scored_results])
                ece_score, ece_diagram = compute_ece(conf_scores, hit_scores)

                # Brier Score: mean((predicted_prob - actual_outcome)^2)
                brier_score = float(np.mean((conf_scores - hit_scores) ** 2))

                # AUROC: requires both classes
                if len(set(hit_scores)) >= 2:
                    auroc_score = float(roc_auc_score(hit_scores, conf_scores))

        fast_ratio = sum(1 for r in results if r["pathway"] == "Fast") / len(results)
        corrective_ratio = sum(1 for r in results if r["pathway"] == "Corrective") / len(results)

        # Bootstrap confidence intervals for key metrics
        ndcg_mean, ndcg_lo, ndcg_hi = bootstrap_ci(ndcgs)
        f1_mean, f1_lo, f1_hi = bootstrap_ci(f1s)
        rouge_mean, rouge_lo, rouge_hi = bootstrap_ci(rouges)

        # Answered-only retrieval metrics (fair comparison for selective systems)
        answered_ndcgs = [r["ndcg_at_3"] for r in answered] if answered else [0.0]
        ndcg_ans_mean, ndcg_ans_lo, ndcg_ans_hi = bootstrap_ci(answered_ndcgs)

        comparison_rows.append({
            "RAG Variant": model_name.upper(),
            "N Answered": len(answered),
            "N Abstained": len(abstained_results),
            # Retrieval
            "NDCG@3": ndcg_mean,
            "NDCG@3 CI": f"[{ndcg_lo}, {ndcg_hi}]",
            "NDCG@3 (Answered)": ndcg_ans_mean,
            "NDCG@3 (Answered) CI": f"[{ndcg_ans_lo}, {ndcg_ans_hi}]",
            "MRR@3": round(float(np.mean(mrrs)), 4),
            "Recall@3": round(float(np.mean(recalls)), 4),
            "Precision@3": round(float(np.mean(precisions)), 4),
            "Hit Rate@3": round(float(np.mean(hit_rates)), 4),
            # Generation (Answered)
            "SemSim (Answered)": round(float(np.mean(semsims)), 4),
            "ROUGE-L (Answered)": rouge_mean,
            "ROUGE-L CI": f"[{rouge_lo}, {rouge_hi}]",
            "BLEU (Answered)": round(float(np.mean(bleus)), 4),
            "F1 (Answered)": f1_mean,
            "F1 CI": f"[{f1_lo}, {f1_hi}]",
            "Exact Match (Answered)": round(float(np.mean(exact_matches)), 4),
            # Hallucination
            "Faithfulness (Answered)": round(float(np.mean(faithfulnesses)), 4),
            "Hallucination Rate (Answered)": round(float(np.mean(halluc_rates)), 4),
            # Confidence
            "ECE": round(float(ece_score), 4),
            "Brier Score": round(float(brier_score), 4),
            "AUROC": round(float(auroc_score), 4),
            # Abstention
            "Abstain Precision": round(abstain_precision, 4),
            "Abstain Recall": round(abstain_recall, 4),
            # Routing
            "Fast %": round(fast_ratio * 100, 1),
            "Corrective %": round(corrective_ratio * 100, 1),
            "Abstain %": round(abstain_rate * 100, 1),
            "Avg Latency (s)": round(float(np.mean(latencies)), 2),
        })

        if ece_diagram:
            pd.DataFrame(ece_diagram).to_csv(
                args.output_dir / f"{model_name}_ece_diagram.csv", index=False
            )

    # =====================================================================
    # Per-Pathway Breakdown Table
    # =====================================================================
    pathway_rows = []
    for variant in args.models:
        variant_results = [r for r in all_runs_results if r["rag_variant"] == variant]
        for pathway in ["Fast", "Corrective", "Abstain", "Direct"]:
            pr = [r for r in variant_results if r["pathway"] == pathway]
            if pr:
                non_abstained = [r for r in pr if not r["abstained"]]
                pathway_rows.append({
                    "Variant": variant.upper(),
                    "Pathway": pathway,
                    "Count": len(pr),
                    "NDCG@3": round(np.mean([r["ndcg_at_3"] for r in pr]), 4),
                    "MRR@3": round(np.mean([r["mrr_at_3"] for r in pr]), 4),
                    "Precision@3": round(np.mean([r["precision_at_3"] for r in pr]), 4),
                    "Hit Rate@3": round(np.mean([r["hit_rate_at_3"] for r in pr]), 4),
                    "Avg F1": round(np.mean([r["f1_score"] for r in non_abstained]), 4) if non_abstained else "N/A",
                    "Avg BLEU": round(np.mean([r["bleu"] for r in non_abstained]), 4) if non_abstained else "N/A",
                    "Avg SemSim": round(np.mean([r["semantic_similarity"] for r in non_abstained]), 4) if non_abstained else "N/A",
                    "Avg Faithfulness": round(np.mean([r["faithfulness"] for r in non_abstained]), 4) if non_abstained else "N/A",
                    "Avg Latency (s)": round(np.mean([r["latency_seconds"] for r in pr]), 2),
                })

    pathway_df = pd.DataFrame(pathway_rows)
    pathway_df.to_csv(args.output_dir / "pathway_breakdown.csv", index=False)

    # =====================================================================
    # Risk-Coverage Analysis (Adaptive only)
    # =====================================================================
    adaptive_results = [r for r in all_runs_results if r["rag_variant"] == "adaptive"]
    if adaptive_results:
        compute_risk_coverage(adaptive_results, args.output_dir)

    # =====================================================================
    # FINAL REPORTS
    # =====================================================================
    summary_df = pd.DataFrame(comparison_rows)
    summary_df.to_csv(args.output_dir / "rag_comparison_summary.csv", index=False)

    print("\n" + "=" * 80)
    print("FINAL COMPARISON SUMMARY REPORT (Test Set)")
    print("=" * 80)
    print(summary_df.to_string(index=False))
    print("=" * 80)

    if len(pathway_rows) > 0:
        print("\nPER-PATHWAY BREAKDOWN (Adaptive Evidence)")
        print("-" * 80)
        print(pathway_df.to_string(index=False))
        print("-" * 80)

    # Ablation study results
    if calibrator is not None and 'ablation_df' in dir():
        print("\nABLATION STUDY (Signal Contribution Evidence)")
        print("-" * 80)
        print(ablation_df.to_string(index=False))
        print("-" * 80)

    # =====================================================================
    # Dynamic Domain Breakdown (Adaptive Variant)
    # =====================================================================
    print("\nCROSS-DOMAIN PERFORMANCE (Adaptive Variant)")
    print("-" * 80)
    domain_rows = []
    
    # Get all unique domains actually present in the test set
    unique_domains = set(r["source_dataset"] for r in adaptive_results)
    
    for domain in unique_domains:
        domain_res = [r for r in adaptive_results if r["source_dataset"] == domain]
        if domain_res:
            domain_ans = [r for r in domain_res if not r["abstained"]]
            domain_rows.append({
                "Domain": domain.split("/")[-1],
                "Count": len(domain_res),
                "NDCG@3": round(np.mean([r["ndcg_at_3"] for r in domain_res]), 4),
                "SemSim": round(np.mean([r["semantic_similarity"] for r in domain_ans]), 4) if domain_ans else 0.0,
                "Abstain %": round((sum(1 for r in domain_res if r["abstained"]) / len(domain_res)) * 100, 1)
            })
            
    if domain_rows:
        domain_df = pd.DataFrame(domain_rows)
        domain_df.to_csv(args.output_dir / "domain_breakdown.csv", index=False)
        print(domain_df.to_string(index=False))
    else:
        print("No adaptive results found to compute domain breakdown.")
    print("-" * 80)

    # Abstention analysis
    abstain_results_final = [r for r in adaptive_results if r["abstained"]]
    if abstain_results_final:
        print("\nABSTENTION ANALYSIS (Failure-Awareness Evidence)")
        print("-" * 80)
        print(f"  Total Abstentions: {len(abstain_results_final)}")
        correct = sum(1 for r in abstain_results_final if r["abstain_was_correct"] == 1)
        incorrect = sum(1 for r in abstain_results_final if r["abstain_was_correct"] == 0)
        unknown = sum(1 for r in abstain_results_final if r["abstain_was_correct"] is None)
        print(f"  Correct Abstentions (would have scored F1 < 0.25): {correct}")
        print(f"  Incorrect Abstentions (could have answered well):  {incorrect}")
        if unknown > 0:
            print(f"  Unknown (no docs for shadow gen):                 {unknown}")
        if abstain_results_final:
            print(f"  Abstain Precision: {correct / len(abstain_results_final):.4f}")
        for r in abstain_results_final[:5]:
            print(f"    Q: {r['question'][:80]}...")
            hyp = r['hypothetical_f1']
            hyp_str = f"{hyp:.3f}" if hyp is not None else "N/A"
            print(f"      Hypothetical F1={hyp_str}, Correct Abstain={r['abstain_was_correct']}")
        print("-" * 80)

    print(f"\nAll outputs saved to: {args.output_dir}")
    print("Files generated:")
    for f in sorted(args.output_dir.glob("*")):
        print(f"  - {f.name}")


if __name__ == "__main__":
    main()

In [ ]:
!python kaggle_adaptive_rag_beir.py \
  --limit 1000 \
  --corpus_size 10000 \
  --models vanilla bm25 hybrid crag adaptive

In [ ]:
!zip -r evaluation_outputs.zip evaluation_outputs/

In [ ]:
import shutil
shutil.make_archive('evaluation_outputs', 'zip', 'evaluation_outputs')